# 03 — Data Integration dan Data Transformation (7 JP)

**Pelatihan Pengolahan Data dengan Spark — TA 2026**

Data latihan bersifat sintetis, bukan data wajib pajak sebenarnya.


**Theory 2 JP; practice 5 JP.** Integrate CSV, JSONL, TXT, Parquet, ORC and PostgreSQL JDBC. A NoSQL integration is discussed as an extension, as no NoSQL service was specified in the supplied Compose environment.


In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
    .appName("SparkTraining-Notebook")
    .master(os.getenv("SPARK_MASTER", "spark://spark-master:7077"))
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4050")
    .config("spark.blockManager.port", "4051")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
print("Spark:", spark.version, "master:", spark.sparkContext.master)


In [ ]:
payments_schema=T.StructType([
 T.StructField("payment_id",T.StringType()),T.StructField("taxpayer_id",T.StringType()),
 T.StructField("payment_date",T.DateType()),T.StructField("amount",T.LongType()),
 T.StructField("channel",T.StringType()),T.StructField("status",T.StringType())])
payments=spark.read.option("header",True).schema(payments_schema).csv("/home/spark/datasets/payments.csv")
filings=spark.read.json("/home/spark/datasets/filings.jsonl")
taxpayers=spark.read.option("header",True).option("inferSchema",True).csv("/home/spark/datasets/taxpayers.csv")
payments.printSchema();filings.printSchema()


In [ ]:
paid=payments.filter(F.col("status")=="PAID")
per_tp=paid.groupBy("taxpayer_id").agg(F.sum("amount").alias("total_paid"))
joined=taxpayers.join(per_tp,"taxpayer_id","left").fillna({"total_paid":0})
joined.select("taxpayer_id","region","total_paid").orderBy(F.desc("total_paid")).show(12,False)


In [ ]:
joined.write.mode("overwrite").parquet("/home/spark/outputs/integration_parquet")
joined.write.mode("overwrite").orc("/home/spark/outputs/integration_orc")
print("Parquet rows:",spark.read.parquet("/home/spark/outputs/integration_parquet").count())
print("ORC rows:",spark.read.orc("/home/spark/outputs/integration_orc").count())
text_df=spark.read.text("/home/spark/datasets/readme.txt");text_df.show(truncate=False)


### JDBC exercise
PostgreSQL credentials are only a local training default. A PostgreSQL JDBC driver is included in the Jupyter image. Spark uses the same jar on executors via the driver-distributed classpath.


In [ ]:
jdbc_url="jdbc:postgresql://postgres:5432/training"
props={"user":"spark","password":"spark","driver":"org.postgresql.Driver"}
reference=spark.read.jdbc(jdbc_url,"taxpayer_reference",properties=props)
reference.show(truncate=False)
reference.join(per_tp,"taxpayer_id","left").show(truncate=False)


In [ ]:
summary=reference.groupBy("region").count()
summary.write.mode("overwrite").jdbc(jdbc_url,"region_reference_summary",properties=props)
print("Saved region_reference_summary")


### Lab tasks
1. Produce valid payment totals by taxpayer, joining taxpayers (left) and payments (filter `PAID`).
2. Deduplicate and union two filtered payment DataFrames (schemas must match).
3. Inspect nested JSON `metadata.batch`, then flatten it.
4. Demonstrate Parquet and ORC round trips and contrast file sizes in `outputs/`.
5. Read one relational table over JDBC; write aggregate results to a new table.
6. Discussion: what is different when the external source is NoSQL?

**Deliverable:** reusable join pipeline with row-count validations.
